In [11]:
import pandas as pd
import numpy as np

TARGETS = ["DeltaTheta", "DeltaX", "DeltaY"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [12]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [13]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
print(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====
        Target  Dataset  Best_Model  Neurons                Best_R2
0   DeltaTheta  Train_1  model_25_1       25   [0.9121630979614301]
1       DeltaX  Train_1  model_14_7       14   [-9.221127308518781]
2       DeltaY  Train_1   model_5_3        5   [0.2717631972933341]
3   DeltaTheta  Train_2   model_3_6        3   [0.8252230541419111]
4       DeltaX  Train_2  model_16_8       16    [-8.96273420141338]
5       DeltaY  Train_2  model_22_5       22   [0.9556355810888035]
6   DeltaTheta      Val   model_9_6        9   [0.9323395565049345]
7       DeltaX      Val   model_5_1        5   [-9.410540558548025]
8       DeltaY      Val   model_6_1        6   [0.3324817146998206]
9   DeltaTheta   Test_1   model_9_6        9   [0.8482544634868923]
10      DeltaX   Test_1  model_14_7       14   [-9.722548723377866]
11      DeltaY   Test_1   model_5_3        5  [0.24119658768888874]
12  DeltaTheta   Test_2  model_21_1       21   [0.8985029456608318]
13